# ⚡ Heat Pump COP Analysis

Deep-dive into the Coefficient of Performance (COP) of the IDM Aero heat pump.

$$COP = \frac{\text{Thermal Energy Output (kWh)}}{\text{Electrical Energy Input (kWh)}}$$

A COP of 3.0 means for every 1 kWh of electricity, the heat pump produces 3 kWh of heat.
Typical air-source heat pumps achieve COP 2.5–4.5 depending on outdoor temperature.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime, timedelta
from sqlalchemy import func

from app.database.database import get_db_session
from app.database.models import Sensor, SensorReading, Location
from app.services.weather import fetch_historical

print("✅ Imports ready")

✅ Imports ready


## 1. Load Heat Pump Sensor Data

Load cumulative meter readings and compute hourly deltas for both electrical input
and thermal output, plus breakdown by heating vs hot water.

In [2]:
SENSOR_NAMES = {
    'elec': 'warmepumpe_Energie_sum',
    'therm_total': 'idm_aero_hp_warmemenge_gesamt',
    'therm_heating': 'idm_aero_hp_warmemenge_heizen',
    'therm_hotwater': 'idm_aero_hp_warmemenge_warmwasser',
    'therm_defrost': 'idm_aero_hp_idm_aero_hp_warmemenge_abtauung',
}

def load_and_resample(sensor_name, freq='1h'):
    """Load cumulative readings and return hourly deltas."""
    with get_db_session() as db:
        sensor = db.query(Sensor).filter(Sensor.name == sensor_name).first()
        readings = (db.query(SensorReading.timestamp, SensorReading.value)
                    .filter(SensorReading.sensor_id == sensor.id)
                    .order_by(SensorReading.timestamp).all())
    df = pd.DataFrame(readings, columns=['timestamp', 'value'])
    df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True)
    ts = df.set_index('timestamp').sort_index()
    hourly = ts['value'].resample(freq).last().interpolate()
    delta = hourly.diff().dropna()
    delta = delta[(delta >= 0) & (delta < delta.quantile(0.995))]
    return delta

data = {}
for key, name in SENSOR_NAMES.items():
    data[key] = load_and_resample(name)
    print(f"{key}: {len(data[key]):,} hourly points")

# Combine into single DataFrame
df_cop = pd.DataFrame({k: v for k, v in data.items()}).dropna()
print(f"\nAligned hourly points: {len(df_cop):,}")

2026-03-24 15:25:04,835 INFO sqlalchemy.engine.Engine select pg_catalog.version()
2026-03-24 15:25:04,836 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-03-24 15:25:04,859 INFO sqlalchemy.engine.Engine select current_schema()
2026-03-24 15:25:04,860 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-03-24 15:25:04,883 INFO sqlalchemy.engine.Engine show standard_conforming_strings
2026-03-24 15:25:04,884 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-03-24 15:25:04,908 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-03-24 15:25:04,911 INFO sqlalchemy.engine.Engine SELECT api_sensors.id AS api_sensors_id, api_sensors.device_id AS api_sensors_device_id, api_sensors.name AS api_sensors_name, api_sensors.description AS api_sensors_description, api_sensors.sensor_type_id AS api_sensors_sensor_type_id, api_sensors.location_id AS api_sensors_location_id, api_sensors.manufacturer AS api_sensors_manufacturer, api_sensors.model AS api_sensors_model, api_sensors.firmware_version AS api_sensors_

## 2. Calculate COP Over Time

Compute hourly, daily, and monthly COP values.

In [3]:
# Hourly COP (filter out near-zero electrical to avoid division issues)
mask = df_cop['elec'] > 0.05
df_cop.loc[mask, 'cop'] = df_cop.loc[mask, 'therm_total'] / df_cop.loc[mask, 'elec']
# Clip extreme COP values (measurement noise)
df_cop['cop'] = df_cop['cop'].clip(upper=10)

# Daily aggregation (sum energy, then divide)
df_daily = df_cop.resample('1D').sum()
df_daily['cop'] = df_daily['therm_total'] / df_daily['elec'].replace(0, np.nan)
df_daily = df_daily.dropna(subset=['cop'])

# Monthly aggregation
df_monthly = df_cop.resample('ME').sum()
df_monthly['cop'] = df_monthly['therm_total'] / df_monthly['elec'].replace(0, np.nan)

print(f"Overall COP: {df_cop['therm_total'].sum() / df_cop['elec'].sum():.2f}")
print(f"Daily COP range: {df_daily['cop'].min():.1f} – {df_daily['cop'].max():.1f}")

# Daily COP trend
fig = px.line(df_daily.reset_index(), x='timestamp', y='cop',
              title='Daily COP Over Time',
              labels={'cop': 'COP', 'timestamp': 'Date'})
fig.add_hline(y=3.0, line_dash='dash', line_color='green',
              annotation_text='Good COP (3.0)')
fig.update_layout(height=400)
fig.show()

Overall COP: 4.33
Daily COP range: 1.6 – 161.7


## 3. COP vs Outdoor Temperature

How efficiently does the heat pump perform at different outdoor temperatures?

In [ ]:
# Fetch historical weather for the full data range
with get_db_session() as db:
    loc = db.query(Location).filter(Location.city == 'Bonn').first()
    LAT, LON = loc.latitude, loc.longitude

start = df_cop.index.min().date()
end = (df_cop.index.max() - timedelta(days=1)).date()
weather = fetch_historical(LAT, LON, start, end)
df_w = pd.DataFrame([{'timestamp': w.timestamp, 'temperature': w.temperature} for w in weather])
df_w['timestamp'] = pd.to_datetime(df_w['timestamp'], utc=True)
df_w = df_w.set_index('timestamp')

# Join COP with temperature
df_cop_temp = df_cop[['cop']].join(df_w['temperature'], how='inner').dropna()

# Bin by temperature
df_cop_temp['temp_bin'] = pd.cut(df_cop_temp['temperature'],
                                  bins=range(-15, 40, 3))
avg_by_temp = df_cop_temp.groupby('temp_bin', observed=True)['cop'].agg(['mean', 'std', 'count']).reset_index()
avg_by_temp['temp_mid'] = avg_by_temp['temp_bin'].apply(lambda x: x.mid)

fig = px.bar(avg_by_temp, x='temp_mid', y='mean', error_y='std',
             title='Average COP by Outdoor Temperature',
             labels={'temp_mid': 'Outdoor Temperature (°C)', 'mean': 'Avg COP'},
             text='count')
fig.add_hline(y=3.0, line_dash='dash', line_color='green')
fig.update_layout(height=450)
fig.show()

2026-03-24 15:25:15,254 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-03-24 15:25:15,256 INFO sqlalchemy.engine.Engine SELECT api_locations.id AS api_locations_id, api_locations.name AS api_locations_name, api_locations.description AS api_locations_description, api_locations.parent_id AS api_locations_parent_id, api_locations.latitude AS api_locations_latitude, api_locations.longitude AS api_locations_longitude, api_locations.altitude AS api_locations_altitude, api_locations.address AS api_locations_address, api_locations.city AS api_locations_city, api_locations.country AS api_locations_country, api_locations.postal_code AS api_locations_postal_code, api_locations.created_at AS api_locations_created_at, api_locations.updated_at AS api_locations_updated_at, api_locations.is_active AS api_locations_is_active 
FROM api_locations 
WHERE api_locations.city = %(city_1)s 
 LIMIT %(param_1)s
2026-03-24 15:25:15,257 INFO sqlalchemy.engine.Engine [generated in 0.00052s] {'city_1': 'Bonn',

DetachedInstanceError: Instance <Location at 0x74b58a5df530> is not bound to a Session; attribute refresh operation cannot proceed (Background on this error at: https://sqlalche.me/e/20/bhk3)

## 4. Energy Breakdown: Heating vs Hot Water vs Defrost

Where is the thermal energy going?

In [ ]:
# Monthly stacked bar — thermal energy breakdown
df_monthly_detail = df_cop[['therm_heating', 'therm_hotwater', 'therm_defrost']].resample('ME').sum()
df_monthly_detail = df_monthly_detail.reset_index()

fig = go.Figure()
for col, color, label in [
    ('therm_heating', '#FF6B35', 'Space Heating'),
    ('therm_hotwater', '#1E90FF', 'Hot Water'),
    ('therm_defrost', '#9370DB', 'Defrost'),
]:
    fig.add_trace(go.Bar(x=df_monthly_detail['timestamp'], y=df_monthly_detail[col],
                         name=label, marker_color=color))

fig.update_layout(barmode='stack', title='Monthly Thermal Energy Breakdown',
                  yaxis_title='kWh', height=450)
fig.show()

# Pie chart — total breakdown
totals = {
    'Space Heating': df_cop['therm_heating'].sum(),
    'Hot Water': df_cop['therm_hotwater'].sum(),
    'Defrost': df_cop['therm_defrost'].sum(),
}
fig2 = px.pie(values=list(totals.values()), names=list(totals.keys()),
              title='Total Thermal Energy Distribution')
fig2.show()

## 5. Monthly COP Trend & Efficiency Rating

Track COP month-over-month with efficiency rating bands.

In [ ]:
df_m = df_monthly.reset_index()

fig = go.Figure()
fig.add_trace(go.Bar(x=df_m['timestamp'], y=df_m['elec'], name='Electrical Input (kWh)',
                     marker_color='steelblue'))
fig.add_trace(go.Bar(x=df_m['timestamp'], y=df_m['therm_total'], name='Thermal Output (kWh)',
                     marker_color='coral'))
fig.add_trace(go.Scatter(x=df_m['timestamp'], y=df_m['cop'], name='COP',
                         mode='lines+markers', yaxis='y2',
                         line=dict(color='green', width=3)))

fig.update_layout(
    title='Monthly Energy & COP',
    yaxis=dict(title='Energy (kWh)'),
    yaxis2=dict(title='COP', overlaying='y', side='right', range=[0, 6]),
    barmode='group', height=500,
)

# Add COP rating bands
for y, label, color in [(4.0, 'Excellent', 'green'), (3.0, 'Good', 'orange'), (2.0, 'Poor', 'red')]:
    fig.add_hline(y=y, line_dash='dot', line_color=color, yref='y2',
                  annotation_text=label, annotation_position='top right')
fig.show()